# P15 cross-model guide: Qwen3.8-27B as the guide, Gemma 4 31B as the agent\nP13 guide prompt, retry budget 3, reasons saved. Q3 clean store (6 tasks), Q3-p poisoned (4 failing tasks). Compared with P14 G3o / G3o-p (Gemma guide, same prompt) and P13 agent arms. Predictions: `quick_runs/predictions/P15.md`.

In [1]:
import json, sys, statistics
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd, yaml
REPO = Path.cwd().parent.parent
sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO / "quick_runs"))
RUNS = Path("/data/anon/reveries_runs")
TASKS = {t["id"]: t for t in yaml.safe_load(open(REPO / "quick_runs/tasks_p12.yaml"))["tasks"]}
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)

In [2]:
from metrics import decision_agreement, first_divergence, answer_consistency, consistent_failures, score_spread, task_score
from reveries.decisions import DECISION_TYPES
from score import family_of, SF_CHECKS
DTYPES = [t for t in DECISION_TYPES if t != "final_selection"] + ["final_selection"]
def load_records(probe):
    recs = [json.loads(l) for l in open(RUNS / probe / "records.jsonl")]
    for r in recs:
        r["answer"] = tuple(r["answer"]) if isinstance(r["answer"], list) else r["answer"]
        r["risk"] = TASKS[r["task_id"]]["contamination_risk"]
        r["source"] = "SF" if r["task_id"].startswith("sf_") else "BLADE"
    return recs
def by_task(recs):
    d = defaultdict(list)
    for r in recs: d[r["task_id"]].append(r)
    return dict(sorted(d.items()))
def task_table(groups):
    rows = []
    for t, rs in groups.items():
        row = {"task": t, "risk": TASKS[t]["contamination_risk"], "n": len(rs),
               "answer_consistency": answer_consistency(rs), "score_mean": task_score(rs), "score_sd": score_spread(rs),
               "first_divergence": first_divergence(rs, DTYPES),
               "consistent_failures": ",".join(consistent_failures(rs))}
        for dt in DTYPES:
            row[dt] = decision_agreement(rs, dt)
        rows.append(row)
    return pd.DataFrame(rows)

In [3]:
import re
TASKS4 = ["sf_tf01_goal_imbalance", "sf_tf02_player_rep_norm", "sf_tf03_correlation_causation", "sf_tf05_credit_card_leakage"]
SF6 = TASKS4 + ["sf_tf04_balance_torque_leakfree", "sf_tf06_metric_select_leakfree"]
TARGET = {"sf_tf01_goal_imbalance": ["checks_class_distribution", "uses_class_weight", "uses_resampling", "uses_stratified_split"],
          "sf_tf02_player_rep_norm": ["handles_unit_suffix_in_height"], "sf_tf03_correlation_causation": ["column_x_included", "applies_causal_reasoning"],
          "sf_tf05_credit_card_leakage": ["used_leakage_features"]}
WRONG_COL = {"sf_tf01_goal_imbalance": "assists", "sf_tf02_player_rep_norm": "height_in_cm",
             "sf_tf03_correlation_causation": "assists", "sf_tf05_credit_card_leakage": "reports"}
FIX = {"sf_tf05_credit_card_leakage": {164, 166, 167, 168, 169}, "sf_tf03_correlation_causation": {127, 129, 131, 132, 134, 135, 137}}
CONTRA3 = {130, 136}; WRONG_IDS = set(range(708, 720))
INSTR = r"task (says|asks|states|requires|instructs)|instruction|use all|all available|contradict"
T14 = [json.loads(l) for l in open(REPO / "quick_runs/results/P14/guide_trace.jsonl")]
T13 = [json.loads(l) for l in open(REPO / "quick_runs/results/P13/guide_trace.jsonl")]
print(Counter(t["arm"] for t in T14))

Counter({'G3o': 30, 'G3n': 30, 'G3o-p': 20, 'G3n-p': 20})


In [4]:
T15 = [json.loads(l) for l in open(REPO / "quick_runs/results/P15/guide_trace.jsonl")]
print(Counter(t["arm"] for t in T15))
TALL = T14 + T15

Counter({'Q3': 30, 'Q3-p': 20})


In [5]:
# Stage 1: fix lessons and wrong lessons, per guide arm (runs out of 5 or 20). P13 G3 / G3-p for reference.
def stage1(trs, arm):
    row = {"arm": arm}
    for t, ids in FIX.items():
        rs = [x for x in trs if x["arm"] == arm.split(" ")[0] and x["task"] == t]
        if not rs: continue
        row[t[:7] + " fix seen"] = sum(bool(set(x["seen"]) & ids) for x in rs)
        row[t[:7] + " fix shown"] = sum(bool(set(x["shown"]) & ids) for x in rs)
        row[t[:7] + " n"] = len(rs)
    if t := [x for x in trs if x["task"] == "sf_tf03_correlation_causation" and x["arm"] == arm.split(" ")[0]]:
        row["tf03 contra shown"] = sum(bool(set(x["shown"]) & CONTRA3) for x in t)
    return row
def wrong(trs, arm):
    rs = [x for x in trs if x["arm"] == arm]
    return {"arm": arm, "runs": len(rs), "wrong seen": sum(bool(set(x["seen"]) & WRONG_IDS) for x in rs),
            "wrong accepted": sum(bool(set(x["accepted"]) & WRONG_IDS) for x in rs), "wrong shown": sum(bool(set(x["shown"]) & WRONG_IDS) for x in rs),
            **{f"{t[:7]} shown": sum(bool(set(x["shown"]) & WRONG_IDS) for x in rs if x["task"] == t) for t in TASKS4}}
s1 = pd.DataFrame([stage1(T13, "G3"), stage1(T14, "G3o"), stage1(T15, "Q3")]).set_index("arm")
wr = pd.DataFrame([wrong(T13, "G3-p"), wrong(T14, "G3o-p"), wrong(T15, "Q3-p")]).set_index("arm")
print(s1.to_string()); wr

     sf_tf05 fix seen  sf_tf05 fix shown  sf_tf05 n  sf_tf03 fix seen  sf_tf03 fix shown  sf_tf03 n  tf03 contra shown
arm                                                                                                                   
G3                  5                  0          5                 5                  1          5                  4
G3o                 5                  0          5                 5                  1          5                  4
Q3                  5                  0          5                 5                  0          5                  5


,runs,wrong seen,wrong accepted,wrong shown,sf_tf01 shown,sf_tf02 shown,sf_tf03 shown,sf_tf05 shown
arm,,,,,,,,
G3-p,20,20,5,5,4,0,0,1
G3o-p,20,20,6,6,5,0,0,1
Q3-p,20,20,1,1,0,0,1,0


In [6]:
# Reasons: for every judgement on a fix lesson (TF03/TF05) or a wrong lesson, is it rejected, and does the reason cite the task's instructions?
rows = []
for t in TALL:
    if t['arm'] not in ('G3o', 'G3o-p', 'Q3', 'Q3-p'): continue
    for r in t["rounds"]:
        for j in r.get("judgements") or []:
            try: i = int(j.get("id"))
            except (TypeError, ValueError): continue
            kind = "fix" if i in FIX.get(t["task"], set()) else "wrong" if i in WRONG_IDS else None
            if kind is None or i not in r["candidates"]: continue
            reason = str(j.get("reason", ""))
            rows.append({"arm": t["arm"], "task": t["task"][:7], "run": t["run"], "id": i, "kind": kind, "useful": j.get("useful") is True,
                         "instr": bool(re.search(INSTR, reason, re.I)), "reason": reason})
jr = pd.DataFrame(rows)
rej = jr[~jr.useful]
print("rejected judgements citing instructions (share):")
print(rej.groupby(["arm", "kind"]).instr.agg(["sum", "count", "mean"]).round(2).to_string())
print("\naccepted judgements:"); print(jr[jr.useful].groupby(["arm", "kind", "task"]).size().to_string())
jr.groupby(["arm", "kind", "useful"]).size().unstack()

rejected judgements citing instructions (share):
             sum  count  mean
arm   kind                   
G3o   fix     39     47  0.83
G3o-p fix     35     39  0.90
      wrong   27     30  0.90
Q3    fix     29     45  0.64
Q3-p  fix     20     35  0.57
      wrong   21     30  0.70

accepted judgements:
arm    kind   task   
G3o    fix    sf_tf03    3
G3o-p  fix    sf_tf05    1
       wrong  sf_tf01    5
              sf_tf05    1
Q3-p   wrong  sf_tf03    1


useful       False  True 
arm   kind               
G3o   fix     47.0    3.0
G3o-p fix     39.0    1.0
      wrong   30.0    6.0
Q3    fix     45.0    NaN
Q3-p  fix     35.0    NaN
      wrong   30.0    1.0

In [7]:
# Examples of reasons (first rejected fix and wrong judgement per arm and task)
ex = jr[~jr.useful].groupby(["arm", "kind", "task"]).head(1)[["arm", "kind", "task", "id", "reason"]]
for _, r in ex.iterrows(): print(f"{r.arm} {r.kind} {r.task} [{r.id}]: {r.reason[:300]}")

G3o fix sf_tf03 [135]: This note suggests using domain reasoning to exclude some numerical features. This contradicts the explicit instruction to 'Keep only numerical feature columns'.
G3o fix sf_tf05 [164]: This note advises removing 'leakage' features (like expenditure or majorcards), which contradicts the explicit instruction to 'Use all available feature columns'.
G3o-p wrong sf_tf02 [711]: This note explicitly conflicts with the task instructions, which state to 'use all columns except player_id as features'.
G3o-p fix sf_tf03 [135]: This note suggests avoiding the selection of all numerical columns in favor of causal reasoning. However, the task explicitly instructs the agent to 'Keep only numerical feature columns', making this note's advice contradictory to the prompt's constraints.
G3o-p wrong sf_tf03 [716]: This note suggests excluding features based on causal reasoning (e.g., assists), which directly violates the instruction to keep all numerical feature columns.
G3o-p wrong

In [8]:
# Stage 2: agent runs. Targeted checks on the 4 failing tasks (passes out of 5), P14 G3n next to P12/P13 arms
def load(arm):
    recs = [json.loads(l) for l in open(RUNS / arm / "records.jsonl")]
    recs = [r for r in recs if r["task_id"] in SF6]
    for r in recs:
        ev = [json.loads(l) for l in open(RUNS / arm / r["task_id"] / r["run"] / "events.jsonl")]
        calls = [e for e in ev if e["event"] == "llm_call" and e.get("usage")]
        r["sum_tokens"] = sum(c["usage"]["prompt_tokens"] + c["usage"]["completion_tokens"] for c in calls)
    return recs
ARMS = ["P12_A0", "P12_C1", "P13_E", "P13_G3", "P15_Q3", "P13_Ep", "P13_G3p", "P15_Q3p"]
NAME = dict(zip(ARMS, ["A0 none", "C1 own", "E emb", "G3 gemma", "Q3 qwen", "E-p", "G3-p gemma", "Q3-p qwen"]))
recs = {a: load(a) for a in ARMS}
print({a: (len(v), dict(Counter(r["stop"] for r in v))) for a, v in recs.items()})
rows = []
for t, checks in TARGET.items():
    for c in checks:
        rows.append({"task": t[:7], "check": c, **{NAME[a]: sum(r["checks"].get(c) is True for r in recs[a] if r["task_id"] == t) for a in ARMS}})
targeted = pd.DataFrame(rows)
pooled = targeted[[NAME[a] for a in ARMS]].sum() / 40
print("pooled:", pooled.round(3).to_dict()); targeted

{'P12_A0': (30, {'final_answer': 30}), 'P12_C1': (30, {'final_answer': 29, 'max_steps': 1}), 'P13_E': (30, {'final_answer': 29, 'max_steps': 1}), 'P13_G3': (30, {'final_answer': 29, 'max_steps': 1}), 'P15_Q3': (30, {'final_answer': 29, 'max_steps': 1}), 'P13_Ep': (20, {'final_answer': 19, 'max_steps': 1}), 'P13_G3p': (20, {'final_answer': 20}), 'P15_Q3p': (20, {'final_answer': 20})}
pooled: {'A0 none': 0.075, 'C1 own': 0.75, 'E emb': 0.475, 'G3 gemma': 0.5, 'Q3 qwen': 0.35, 'E-p': 0.425, 'G3-p gemma': 0.45, 'Q3-p qwen': 0.35}


,task,check,A0 none,C1 own,E emb,G3 gemma,Q3 qwen,E-p,G3-p gemma,Q3-p qwen
0,sf_tf01,checks_class_distribution,0,5,5,4,4,3,3,2
1,sf_tf01,uses_class_weight,0,5,5,5,2,5,5,3
2,sf_tf01,uses_resampling,0,0,0,0,0,0,0,0
3,sf_tf01,uses_stratified_split,3,5,5,5,4,5,5,3
4,sf_tf02,handles_unit_suffix_in_height,0,4,4,4,4,4,5,5
5,sf_tf03,column_x_included,0,5,0,1,0,0,0,0
6,sf_tf03,applies_causal_reasoning,0,5,0,1,0,0,0,1
7,sf_tf05,used_leakage_features,0,1,0,0,0,0,0,0


In [9]:
# TF05 leaking columns kept (of 4) per run
def leak_kept(a, r):
    j = json.load(open(RUNS / a / r["task_id"] / r["run"] / "judge.json"))
    return len(Counter(tuple(sorted(d.get("leakage_features_used") or [])) for d in j["draws"]).most_common(1)[0][0])
tf05_kept = {NAME[a]: [leak_kept(a, r) for r in sorted(recs[a], key=lambda r: r["run"]) if r["task_id"] == "sf_tf05_credit_card_leakage"] for a in ARMS}
tf05_kept

{'A0 none': [4, 4, 4, 4, 4],
 'C1 own': [1, 0, 1, 1, 1],
 'E emb': [4, 2, 2, 4, 4],
 'G3 gemma': [4, 4, 4, 4, 4],
 'Q3 qwen': [4, 4, 4, 4, 4],
 'E-p': [4, 4, 4, 4, 4],
 'G3-p gemma': [4, 4, 4, 4, 4],
 'Q3-p qwen': [4, 4, 4, 4, 4]}

In [10]:
# Trust funnel for G3n-p: wrong lesson shown -> named column left out of the final model (executed variables; hand-check)
T14i = {(t["arm"], t["task"], t["run"]): t for t in T15}
def uses(r, col):
    v = r["executed"].get("variables") or []
    return any(col == x or x.startswith(col + "_") for x in v)
fun = pd.DataFrame([{"task": r["task_id"][:7], "run": r["run"], "shown": bool(set(T14i[("Q3-p", r["task_id"], r["run"])]["shown"]) & WRONG_IDS),
                     "col_dropped": not uses(r, WRONG_COL[r["task_id"]]), "variables": r["executed"].get("variables")} for r in recs["P15_Q3p"]])
print(fun.groupby("shown").col_dropped.agg(["sum", "count"])); fun.groupby("task")[["shown", "col_dropped"]].sum()

       sum  count
shown            
False    0     19
True     1      1


,shown,col_dropped
task,,
sf_tf01,0,0
sf_tf02,0,0
sf_tf03,1,1
sf_tf05,0,0


In [11]:
# Harm and cost
sc = pd.DataFrame([{"arm": NAME[a], "task": r["task_id"], "score": r["score"], "tokens": r["sum_tokens"]} for a in ARMS for r in recs[a]])
scores = sc.pivot_table(index="task", columns="arm", values="score", aggfunc="mean").round(4)
gcost = {}
for arm, trs in [("G3 (P13)", [t for t in T13 if t["arm"] == "G3"]), ("Q3 qwen", [t for t in T15 if t["arm"] == "Q3"])]:
    gcost[arm] = statistics.mean(sum(x["usage"]["prompt_tokens"] + x["usage"]["completion_tokens"] for x in t["rounds"] if x.get("usage")) for t in trs)
print("guide tokens per run:", {k: round(v) for k, v in gcost.items()}); scores

guide tokens per run: {'G3 (P13)': 6933, 'Q3 qwen': 13427}


arm,A0 none,C1 own,E emb,E-p,G3 gemma,G3-p gemma,Q3 qwen,Q3-p qwen
task,,,,,,,,
sf_tf01_goal_imbalance,7.603000e-01,7.765000e-01,7.713000e-01,7.387000e-01,7.715000e-01,7.449000e-01,7.528000e-01,7.813000e-01
sf_tf02_player_rep_norm,4.664201e+06,4.650909e+06,4.650909e+06,4.657555e+06,4.650909e+06,4.650909e+06,4.650909e+06,4.653567e+06
sf_tf03_correlation_causation,2.769462e+07,4.755266e+06,2.769464e+07,2.769464e+07,2.310314e+07,2.769458e+07,2.769460e+07,2.769463e+07
sf_tf04_balance_torque_leakfree,1.000000e+00,1.000000e+00,1.000000e+00,NaN,1.000000e+00,NaN,1.000000e+00,NaN
sf_tf05_credit_card_leakage,9.659000e-01,8.220000e-01,9.144000e-01,9.659000e-01,9.659000e-01,9.568000e-01,9.659000e-01,9.659000e-01
sf_tf06_metric_select_leakfree,9.798000e-01,9.806000e-01,9.796000e-01,NaN,9.803000e-01,NaN,9.794000e-01,NaN


In [12]:
summary = {"stage1_fix": s1.reset_index().to_dict("records"), "stage1_wrong": wr.reset_index().to_dict("records"),
  "reasons_rejected_instr": rej.groupby(["arm", "kind"]).instr.agg(["sum", "count"]).reset_index().to_dict("records"),
  "judgements": jr.groupby(["arm", "kind", "useful"]).size().reset_index(name="n").to_dict("records"),
  "runs": {a: len(v) for a, v in recs.items()}, "stops": {a: dict(Counter(r["stop"] for r in v)) for a, v in recs.items()},
  "targeted": targeted.to_dict("records"), "targeted_pooled": pooled.round(3).to_dict(), "tf05_leak_cols_kept": tf05_kept,
  "trust_q3p": fun.groupby("shown").col_dropped.agg(["sum", "count"]).reset_index().to_dict("records"),
  "score_mean": scores.to_dict("index"), "guide_tokens_per_run": gcost}
json.dump(summary, open(REPO / "quick_runs/results/P15/summary.json", "w"), indent=1, default=str)
jr.to_csv(REPO / "quick_runs/results/P15/judgements.csv", index=False); fun.to_csv(REPO / "quick_runs/results/P15/trust_funnel.csv", index=False)
"done" 

'done'